In [ ]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

In [ ]:
#mostra il numero di righe totali del file:
!wc -l fhvhv_tripdata_2021-01.csv

In [ ]:
#si crea un Dataframe in cui si mettono i dati del file csv.
#con .option() si vuole mostrare lo schema del dataset 

df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv') 

In [ ]:
df.schema

In [ ]:
#si crea un nuovo file di nome: head.csv 
#i dati inseriti in quel .csv sono le prime 1000 righe del file csv originale
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [ ]:
df_head = spark.read.csv('head.csv', header=True)

In [ ]:
import pandas as pd 

In [ ]:
#qui si utilizza Pandas come Framework 

df_pandas = pd.read_csv('head.csv')

In [ ]:
#mostra lo schema del DataFrame in Pandas
df_pandas.dtypes

In [ ]:
#si crea un Dataframe in Spark, a partire dal DataFrame in Pandas
df_spark = spark.createDataFrame(df_pandas)

In [ ]:
from pyspark.sql import types

In [ ]:
#abbiamo imposto questa schema alla variabile 'schema'. 
#Lo schema l'ho preso da df_spark.schema di 2 celle sopra
schema = types.StructType(
    [types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)]
)

In [ ]:
#per questo Dataframe si impone lo schema 
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv') 

In [ ]:
df.head(5)

In [ ]:
#Il file .csv al momento è come un unico grande file, lavorare con questo file cosi pesante
# è un problema, quindi si va suddividere il file i 24 partitions, cosi che Spark
#che lavora in pararello, può lavorare con 24 file più piccoli, invece di un unico grande file.

df = df.repartition(24)

In [ ]:
#creamo i file parquet a partire dal Dataframe df che è stato diviso in 24 partitions, 
# quindi si creeranno 24 file parquet, invece di un unico grande file parquet.

df.write.parquet('fhvhv/2021/01/')

Video 5: Spark DataFrames

In [ ]:
#lettura dei file parquet creati nel video precedente 

df = spark.read.parquet('fhvhv/2021/01/')

In [ ]:
df.printSchema()

#NOTA: lo schema nei file Parquet è ottimizzato e scritto in modo efficiente, 
# quindi è più veloce da leggere rispetto al file CSV originale, che invece è un formato di testo non strutturato.

operazion di manipolazione dei dati del Dataframe: 
- .select() per selezionare solo alcune colonne del Dataframe
- .filter() per filtrare i dati del Dataframe in base a una condizione    
- .groupBy() per raggruppare i dati del Dataframe in base a una o più colonne

NOTA: l'operazione .select() e .filter() non hanno eseguito niente...
 
sono delle **Transformation**, quindi sono "Lazy", hanno creato il piano di esecuzione (DAG)

con .show() (**Actions**) è partita l'esecuzione del piano ed eseguite le 2 operazioni. 

In [ ]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003').show(5)

In [ ]:
#import delle Functions di Spark, per poter utilizzare le funzioni di Spark SQL
from pyspark.sql import functions as F

In [ ]:
#si creano 2 nuove colonne e si impone il tipo alla colonna con F.to_date (Function di Spark SQL)

df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)).show(5)

Sintatti di *df.withColumn('nomeColonnaNuova', F.nomeFunzione(df.nomeColonnaOriginale))*
questo metodo è molto utilizzato in Spark.

In [ ]:
#Invece di aggiungere 2 colonne alla fine, si potrebbe sovrascrivere le colonne 
#però questo non è un buon modo di lavorare perchè si perderebbero i dati originali, 
# quindi è meglio creare nuove colonne, invece di sovrascrivere quelle vecchie.

#Quello che si può fare è creare una select e specificare solo le colonne che ci interessano, 
#quindi si prendono le colonne che abbiamo definito noi al posto di quelle originali.

df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID').show(5)

In [ ]:
#si crea un metodo: (Python Function)
#esempio di base_num: B02872 
def crazy_stuff(base_num): 
    num = int(base_num[1:]) #salta il primpo carattere 

    if num % 7 == 0: 
        return f's/{num:03x}'
    elif num % 3 == 0: 
        return f'a/{num:03x}'
    else: 
        return f'e/{num:03x}'

In [ ]:
 #chiamo il metodo di sopra: 
crazy_stuff('B02871')

In [ ]:
#si crea una UDF (User Defined Function) per poterla utilizzare con Spark SQL
#NOTA: l'UDF utilizza la funzione definita sopra: crazy_stuff, 
# e si impone il tipo di ritorno della funzione con returnType=types.StringType()
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [ ]:
df.withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id','pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID').show(5)